In [1]:
import boto3
import math
from sagemaker import get_execution_role
from pprint import pprint
import time

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[02/05/25 23:02:03] INFO     Found credentials from IAM Role:                                   ]8;id=96898;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=822063;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Functions

In [2]:
def get_specs(str_instance):
    if str_instance == 'm5.large':
        int_vcpu = 2
        int_memory_gb = 8
    elif str_instance == 'm5.xlarge':
        int_vcpu = 4
        int_memory_gb = 16
    elif str_instance == 'm5.2xlarge':
        int_vcpu = 8
        int_memory_gb = 32
    elif str_instance == 'm5.4xlarge':
        int_vcpu = 16
        int_memory_gb = 64
    elif str_instance == 'm5.8xlarge':
        int_vcpu = 32
        int_memory_gb = 128
    elif str_instance == 'm5.12xlarge':
        int_vcpu = 48
        int_memory_gb = 192
    int_memory_mebibytes = math.ceil(int_memory_gb * 953.674)
    dict_output = {
        'int_vcpu': int_vcpu,
        'int_memory_gb': int_memory_gb,
        'int_memory_mebibytes': int_memory_mebibytes,
    }
    return dict_output

### Constants

In [3]:
str_image_name = 'genxii-early-30-90'
int_iteration = 1
str_instance = 'm5.4xlarge'
dict_specs = get_specs(str_instance=str_instance)
int_vcpu = dict_specs['int_vcpu']
int_memory_gb = dict_specs['int_memory_gb']
int_memory_mebibytes = dict_specs['int_memory_mebibytes']
for key, val in dict_specs.items():
    print(f'{key}: {val}')

int_vcpu: 16
int_memory_gb: 64
int_memory_mebibytes: 61036


### Create compute environment

In [4]:
# initialize class
cls_client = boto3.client('batch')

[02/05/25 23:02:07] INFO     Found credentials from IAM Role:                                   ]8;id=43946;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=72681;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [5]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [6]:
# create compute environment
while True:
    try:
        str_compute_env_name = f'env-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_compute_environment(
            computeEnvironmentName=str_compute_env_name,
            type= 'Managed', 
            state= 'ENABLED',
            serviceRole = str_role,
            computeResources={
                #'type': 'SPOT',
                'type': 'EC2',
                'minvCpus': 0,
                'maxvCpus': 256, 
                'desiredvCpus': int_vcpu,
                'instanceTypes': [
                    str_instance,
                ], 
                'subnets': ['subnet-044e573651bb251a7'], 
                'securityGroupIds': ['sg-03904237048cdc335'], 
                'instanceRole': 'ecsInstanceRole',
                #'spotIamFleetRole': 'AmazonEC2SpotFleetTaggingRole',
            },
        )
        pprint(dict_response)
        break
    except:
        int_iteration += 1

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '161',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 05 Feb 2025 23:02:08 GMT',
                                      'x-amz-apigw-id': 'FiIdmFXIPHcEErQ=',
                                      'x-amzn-requestid': 'f7d84a8b-4c34-4fff-b3ba-e7d9594e95c6',
                                      'x-amzn-trace-id': 'Root=1-67a3edf0-43e6a1ee0ccf471325fe3f9a'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'f7d84a8b-4c34-4fff-b3ba-e7d9594e95c6',
                      'RetryAttempts': 0},
 'computeEnvironmentArn': 'arn:aws:batch:

### Create Job Queue

In [7]:
# create job queue (this is where AWS will store your jobs until an EC2 Instance is available to run them)
while True:
    try:
        str_job_queue_name = f'queue-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_job_queue(
            jobQueueName=str_job_queue_name,
            state='ENABLED',
            priority=1,
            computeEnvironmentOrder=[
                {
                    'order': 1,
                    'computeEnvironment': str_compute_env_name,
                },
            ]
        )
        # get arn
        str_job_queue_arn = dict_response['jobQueueArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '135',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 05 Feb 2025 23:02:30 GMT',
                                      'x-amz-apigw-id': 'FiIg_HH6PHcEvpw=',
                                      'x-amzn-requestid': 'ec9c56cd-834d-4e89-b2a3-62e1b733f1b6',
                                      'x-amzn-trace-id': 'Root=1-67a3ee05-0c3738b95449c59c31db96fc'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'ec9c56cd-834d-4e89-b2a3-62e1b733f1b6',
                      'RetryAttempts': 0},
 'jobQueueArn': 'arn:aws:batch:us-west-2:

### Register job definition

In [8]:
# job definition
while True:
    try:
        str_job_definition = f'job-def-{str_image_name}-{int_iteration}'
        dict_response = cls_client.register_job_definition(
            type='container',
            containerProperties={
                'image': f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_image_name}:latest',
                'memory': int_memory_mebibytes,
                'vcpus': int_vcpu,
            },
            jobDefinitionName=str_job_definition,
        )
        # get arn
        str_job_def_arn = dict_response['jobDefinitionArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '169',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 05 Feb 2025 23:02:30 GMT',
                                      'x-amz-apigw-id': 'FiIhAF30vHcEpjA=',
                                      'x-amzn-requestid': '74de8a52-d5ea-4b60-875d-d8e85829be53',
                                      'x-amzn-trace-id': 'Root=1-67a3ee06-229152c14f13a58846e38f20'},
                      'HTTPStatusCode': 200,
                      'RequestId': '74de8a52-d5ea-4b60-875d-d8e85829be53',
                      'RetryAttempts': 0},
 'jobDefinitionArn': 'arn:aws:batch:us-we

### Submit job

In [12]:
# # submit a job (only for testing)
# while True:
#     try:
#         str_job_name = f'job-name-{str_image_name}-{int_iteration}'
#         response = cls_client.submit_job(
#             jobDefinition=str_job_definition,
#             jobQueue=str_job_queue_name,
#             jobName=str_job_name,
#         )
#         pprint(response)
#         break
#     except:
#         time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '179',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 06 Feb 2025 00:01:38 GMT',
                                      'x-amz-apigw-id': 'FiRLfGz5vHcEdAA=',
                                      'x-amzn-requestid': 'b3e89150-67d8-40da-8869-0a5a55fa52ae',
                                      'x-amzn-trace-id': 'Root=1-67a3fbe2-66b6903313847f5b36e0f286'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'b3e89150-67d8-40da-8869-0a5a55fa52ae',
                      'RetryAttempts': 0},
 'jobArn': 'arn:aws:batch:us-west-2:83669

### Show arns

In [10]:
print(f'Job Queue ARN: {str_job_queue_arn}')
print(f'Job Definition ARN: {str_job_def_arn}')

Job Queue ARN: arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-early-30-90-1
Job Definition ARN: arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-early-30-90-1:1
